In [ ]:
import math
import random

random.seed(42)

def euclidean_distance(a: tuple, b: tuple) -> float:
    """
    Computes the euclidean distance between two 2D points
    
    Args:
        a (tuple): _description_
        b (tuple): _description_

    Returns:
        float: _description_
    """    
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

def tour_length(tour, dist):
    """
    Computes the length of a generated tsp tour(solution).

    Args:
        tour (_type_): _description_
        dist (_type_): _description_

    Returns:
        _type_: _description_
    """    
    n = len(tour)
    return sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))

def load_tsp(filename: str):
    """
    Loads the tsp coordinates and also computes the distances between them, also it checks for duplicate instances
    in the dataset and removes and duplicates.

    Args:
        filename (str): _description_

    Returns:
        _type_: _description_
    """    
    coords_raw = []
    with open(filename) as f:
        for line in f:
            if line.strip() == 'NODE_COORD_SECTION':
                break
        for line in f:
            if line.strip() == 'EOF':
                break
            parts = line.split()
            x, y = float(parts[1]), float(parts[2])
            coords_raw.append((x, y))

    seen = {}
    coords = []
    for coord in coords_raw:
        if coord not in seen:
            seen[coord] = len(coords)
            coords.append(coord)

    n = len(coords)

    dists = [[euclidean_distance(coords[i], coords[j])
              for j in range(n)]
             for i in range(n)]

    return coords, dists

In [ ]:
import time
def ant_colony_system_tsp(filename, iterations=500, num_ants=20,
                           alpha=1, beta=2,
                           q0=0.9,
                           evaporation_rate=0.1,
                           local_evaporation_rate=0.1):
    """
    Implementation of the ant colony system designed to solve the tsp problem.

    Args:
        filename (_type_): input file for data for the tsp problem
        iterations (int, optional): Nr of iterations until stop condition. Defaults to 500.
        num_ants (int, optional): Nr of ants that is trying to find the optimum. Defaults to 20.
        alpha (int, optional): Controls how much weight does the pheromone have in deciding what is the next node. Defaults to 1.
        beta (int, optional): Controls how much weight does the visibility have in deciding what is the next node. Defaults to 2.
        q0 (float, optional): The exploitation vs exploration parameter. Defaults to 0.9.
        evaporation_rate (float, optional): The rate with which pheromones evaporate off edges globally. Defaults to 0.1.
        local_evaporation_rate (float, optional): Controls pheoromone update balance after each ant run. Defaults to 0.1.

    Returns:
        best_tour_length, duration_time (float, float): The best result found for the problem and the time it took to find it in seconds.
    """    
    
    start_time = time.time()
    
    coords, distances = load_tsp(filename)
    num_cities = len(coords)

    visited = [False] * num_cities
    nn_tour = [0]
    visited[0] = True
    for _ in range(num_cities - 1):
        current = nn_tour[-1]
        nearest = min(
            [c for c in range(num_cities) if not visited[c]],
            key=lambda c: distances[current][c]
        )
        nn_tour.append(nearest)
        visited[nearest] = True
    nn_tour_length = tour_length(nn_tour, distances)
    initial_pheromone = 1.0 / (num_cities * nn_tour_length)  

    pheromone_matrix = [[initial_pheromone for _ in range(num_cities)]
                        for _ in range(num_cities)]
    
    visibility_matrix = [[0.0 if city_i == city_j else 1.0 / distances[city_i][city_j]
                          for city_j in range(num_cities)]
                         for city_i in range(num_cities)]

    global_best_tour = nn_tour.copy()
    global_best_length = nn_tour_length


    for iteration in range(iterations):

        iteration_tours = []
        iteration_tour_lengths = []

        for ant in range(num_ants):

            visited_cities = [random.randint(0, num_cities - 1)]
            current_city = visited_cities[0]

            for _ in range(num_cities - 1):
                unvisited_cities = [city for city in range(num_cities)
                                    if city not in visited_cities]

                q = random.random()

                if q <= q0:
                    next_city = max(
                        unvisited_cities,
                        key=lambda city: (pheromone_matrix[current_city][city] ** alpha)
                                       * (visibility_matrix[current_city][city] ** beta)
                    )
                else:
                    transition_weights = [
                        (pheromone_matrix[current_city][candidate_city] ** alpha)
                        * (visibility_matrix[current_city][candidate_city] ** beta)
                        for candidate_city in unvisited_cities
                    ]
                    total_weight = sum(transition_weights)

                    if total_weight == 0:
                        next_city = random.choice(unvisited_cities)
                    else:
                        selection_probs = [w / total_weight for w in transition_weights]
                        spin = random.random()
                        cumulative_prob = 0
                        next_city = unvisited_cities[-1]
                        for idx, candidate_city in enumerate(unvisited_cities):
                            cumulative_prob += selection_probs[idx]
                            if spin <= cumulative_prob:
                                next_city = candidate_city
                                break

                pheromone_matrix[current_city][next_city] = (
                    (1 - local_evaporation_rate) * pheromone_matrix[current_city][next_city]
                    + local_evaporation_rate * initial_pheromone
                )
                pheromone_matrix[next_city][current_city] = (
                    pheromone_matrix[current_city][next_city]  
                )

                visited_cities.append(next_city)
                current_city = next_city

            ant_tour_length = tour_length(visited_cities, distances)
            iteration_tours.append(visited_cities)
            iteration_tour_lengths.append(ant_tour_length)

        iteration_best_idx = min(range(num_ants), key=lambda k: iteration_tour_lengths[k])
        iteration_best_tour = iteration_tours[iteration_best_idx]
        iteration_best_length = iteration_tour_lengths[iteration_best_idx]

        if iteration_best_length < global_best_length:
            global_best_length = iteration_best_length
            global_best_tour = iteration_best_tour.copy()

        for city_i in range(num_cities):
            for city_j in range(num_cities):
                pheromone_matrix[city_i][city_j] *= (1 - evaporation_rate)
                pheromone_matrix[city_i][city_j] = max(
                    pheromone_matrix[city_i][city_j], 1e-10
                )

        global_best_deposit = evaporation_rate * (1.0 / global_best_length)
        for step in range(num_cities):
            city_from = global_best_tour[step]
            city_to = global_best_tour[(step + 1) % num_cities]
            pheromone_matrix[city_from][city_to] += global_best_deposit
            pheromone_matrix[city_to][city_from] += global_best_deposit  

    dur_time = time.time() - start_time
    
    return global_best_length, dur_time

In [30]:
print(ant_colony_system_tsp("../Lab3Assigment3/rw1621.tsp", iterations=1, num_ants=1))

(32276.665616157425, 4.557023763656616)


In [ ]:
def generate_report(input, output, iterations, ants) :
    """
    Function that generates the report for each of the 2 tsp instances we have.

    Args:
        input (_type_): the tsp problem we are trying to solve
        output (_type_): the output file where all runs are logged
        iterations (_type_): a list containing different values for iterations
        ants (_type_): a list containing different values for iterations
    """    
    with open(output, 'a') as f:
        f.write(f"\n____Report for {output} STARTED______\n")
        
    
    for iteration in iterations :
        for ant in ants :
            with open(output, 'a') as f :
                result, dur_time = ant_colony_system_tsp(input, iterations=iteration, num_ants=ant)
                f.write(f"\nNr_of_iterations={iteration}, nr_of_ants={ant}, result={result}, time={dur_time}s\n")
                
    with open(output, 'a') as f:
        f.write(f"\n____Report for {output} is OVER______\n")

In [32]:
generate_report('../Lab3Assigment3/berlin52.tsp',
                'berlin_report.txt',
                [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000],
                [1, 2, 5, 10, 20, 26, 52])

In [5]:
generate_report('../Lab3Assigment3/rw1621.tsp', 'rwanda_report.txt',
                [1, 2, 5, 8, 10],
                [1, 2, 5, 8, 10])